<a href="https://colab.research.google.com/github/saigoncanhquan/VoiceStudio/blob/main/VoiceStudio_Colab_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ VoiceStudio & OmniVoice — GPU Server trên Google Colab

Notebook này biến Google Colab (với **GPU NVIDIA T4 miễn phí**) thành máy chủ xử lý AI siêu tốc cho ứng dụng **VoiceStudio / OmniVoice** chạy trên máy tính của bạn.

### ⚡ Hướng dẫn sử dụng:
1. Chọn menu: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
2. Bấm vào nút **Run all** (hoặc nhấn `Ctrl + F9`).
3. Sau khoảng 1-2 phút, kéo xuống cuối cell sẽ thấy dòng link công khai: `https://xxxx.gradio.live`.
4. Sao chép link đó và dán vào ô **Google Colab Gradio Live URL** trên WebUI ở máy tính cá nhân của bạn!

In [1]:
# 1. Cài đặt các thư viện cần thiết trên Colab GPU
!pip install -q soundfile gradio torchaudio accelerate transformers
!git clone https://github.com/saigoncanhquan/OmniVoice.git /content/OmniVoice_Repo
%cd /content/OmniVoice_Repo
!pip install -q -e .

Cloning into '/content/OmniVoice_Repo'...
remote: Enumerating objects: 532, done.
remote: Counting objects: 100% (308/308), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 532 (delta 208), reused 181 (delta 168), pack-reused 224 (from 1)
Receiving objects: 100% (532/532), 2.36 MiB | 2.02 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/content/OmniVoice_Repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for omnivoice (pyproject.toml) ... done


In [2]:
# 2. Khởi tạo máy chủ OmniVoice TikTok Engine trên Colab GPU
import os, sys, shutil, time
from datetime import datetime
import torch
import gradio as gr
from omnivoice.tiktok_tts_engine import TikTokTTSEngine

print("Đang nạp mô hình OmniVoice lên GPU:", torch.cuda.get_device_name(0))
engine = TikTokTTSEngine(temp_dir="/content/temp_omnivoice")
print("✓ Nạp mô hình lên GPU thành công!")

def process_tts(
    text,
    voice_choice,
    custom_voice,
    speed=1.15,
    pause_duration=0.08,
    pitch_consistent=True,
    num_steps=24,
    guidance_scale=3.0
):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_basename = f"colab_gen_{timestamp}"

    # Định vị file mẫu
    ref_audio = None
    if voice_choice == "Giọng Nam TikTok (Chuẩn AI sạch)":
        voice_mode = "design_male"
    elif voice_choice == "Giọng Nữ TikTok (Chuẩn AI sạch)":
        voice_mode = "design_female"
    elif voice_choice == "Giọng Nam Clone Nhanh 4s (Tối ưu CPU)":
        voice_mode = "clone"
        ref_audio = "/content/OmniVoice_Repo/VoiceTemplate/giong_nam_ref_4s.wav"
    elif voice_choice == "Giọng Nam Clone (từ GiọngNam.wav)":
        voice_mode = "clone"
        ref_audio = "/content/OmniVoice_Repo/VoiceTemplate/GiọngNam.wav"
    elif voice_choice == "Upload Giọng Mẫu Riêng (Custom Clone)":
        voice_mode = "clone"
        ref_audio = custom_voice
    else:
        voice_mode = "design_male"

    t0 = time.time()
    result = engine.synthesize_script(
        script_text=text,
        voice_mode=voice_mode,
        ref_audio_path=ref_audio,
        speed=float(speed),
        pause_between_chunks=float(pause_duration),
        num_step=int(num_steps),
        guidance_scale=float(guidance_scale),
        pitch_consistency=bool(pitch_consistent),
        output_basename=out_basename
    )
    t_elapsed = time.time() - t0

    audio_path = result["audio_path"]
    srt_path = result["srt_path"]
    duration = result.get("duration", 0.0)
    segments = result.get("segments", [])

    srt_content = ""
    if os.path.exists(srt_path):
        with open(srt_path, "r", encoding="utf-8") as f:
            srt_content = f.read()

    info_text = f"Thời lượng âm thanh: {duration:.2f}s | GPU T4 xử lý trong: {t_elapsed:.2f}s (Nhanh gấp {duration/t_elapsed:.1f}x)"
    return audio_path, srt_content, srt_path, audio_path, info_text

# 3. Khởi tạo giao diện Gradio với Public Live Link
demo = gr.Interface(
    fn=process_tts,
    inputs=[
        gr.Textbox(label="Script"),
        gr.Dropdown(choices=[
            "Giọng Nam TikTok (Chuẩn AI sạch)",
            "Giọng Nữ TikTok (Chuẩn AI sạch)",
            "Giọng Nam Clone Nhanh 4s (Tối ưu CPU)",
            "Giọng Nam Clone (từ GiọngNam.wav)",
            "Upload Giọng Mẫu Riêng (Custom Clone)"
        ], value="Giọng Nam TikTok (Chuẩn AI sạch)"),
        gr.Audio(type="filepath"),
        gr.Slider(0.9, 1.4, value=1.15),
        gr.Slider(0.02, 0.4, value=0.08),
        gr.Checkbox(value=True),
        gr.Slider(16, 40, value=24),
        gr.Slider(1.0, 5.0, value=3.0)
    ],
    outputs=[
        gr.Audio(label="Audio Output"),
        gr.Textbox(label="SRT Content"),
        gr.File(label="SRT File"),
        gr.File(label="Audio File"),
        gr.Textbox(label="Info")
    ],
    title="VoiceStudio Colab GPU Server"
)

demo.launch(share=True)

Đang nạp mô hình OmniVoice lên GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✓ Nạp mô hình lên GPU thành công!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b51f7c8ed7a53b8db1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
